# In-Class Exercises 5 (solution)

> **Note:**
> 
> Please commit every time you solve one of the exercises. An example commit message
> could be `"Solution to question 1"`. Feel free to commit more than once per
> exercise if solving it requires multiple complicated steps.

## Imports and paths as in Exercise 4

In [ ]:
from pathlib import Path

import pandas as pd

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = False
pd.options.plotting.backend = "plotly"


this_dir = Path()
this_dir.resolve()
data_file = this_dir.resolve() / "original_data" / "Data_Elections.dta"

## Functional data management

---
### Task 1

The next cell contains the data management code from the solution in exercise 4. For
convenience, we have only kept those parts that actually do something with the data.
That is, all code for looking at features interactively has been removed.

Bring it into a form that is similar to what you saw in the screencast on functional
data management.

> **Note:** Make sure you do not get any name clashes between this code and the code you
> will write. That is, re-use `data_file`, but pick different names for anything else
> you may need in the global namespace.

In [ ]:
data = pd.read_stata(data_file, convert_categoricals=False)
data_info = pd.io.stata.StataReader(data_file)

data = data.drop(columns=["cid", "id"])
data = data.rename(
    columns={
        "Country": "country",
        "Nuts_id": "nuts_id",
        "Name": "nuts_name",
        "Year": "year",
        "ElectionType": "election_type",
        "EligibleVoters": "number_eligible_voters",
        "Valid": "number_valid_votes",
        "HHI": "number_parties_effective",
        "Far_Right": "number_votes_far_right",
        "Far_Left": "number_votes_far_left",
        "Far_Right_share": "share_votes_far_right",
        "Far_Left_share": "share_votes_far_left",
        "Far_share": "share_votes_far_any",
        "Turnout": "share_voter_turnout",
        "F0Far_Incumbent": "number_votes_far_any_incumbent",
        "left": "pm_party_left",
    },
)
for col in "country", "nuts_id", "nuts_name":
    data[col] = data[col].astype(pd.CategoricalDtype())
data["year"] = data["year"].astype(pd.Int16Dtype())
election_cats = data_info.value_labels()["ElectionType"].copy()
for i, to_append in (5, " A"), (7, " B"):
    election_cats[i] += to_append
election_type = (
    data["election_type"].astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
)
election_type = election_type.cat.rename_categories(election_cats)
data["election_type"] = election_type
for col in data.columns:
    if col.startswith("number_") and col != "number_parties_effective":
        data[col] = data[col].round().astype(pd.UInt32Dtype())
pm_party_orientation = (
    data["pm_party_left"].astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
)
pm_party_orientation = pm_party_orientation.cat.rename_categories(
    {0: "Other", 1: "Left-leaning"},
)
data["pm_party_orientation"] = pm_party_orientation
data = data.drop(columns="pm_party_left")

In [ ]:
data

In [ ]:


def clean_data(raw, metadata):
    df = pd.DataFrame(index=raw.index)
    for old, new in {
        "Country": "country",
        "Nuts_id": "nuts_id",
        "Name": "nuts_name",
    }.items():
        df[new] = raw[old].astype(pd.CategoricalDtype())
    df["year"] = raw["Year"].astype(pd.Int16Dtype())
    df["election_type"] = _convert_election_cats(
        raw_sr=raw["ElectionType"],
        labels_dict=metadata.value_labels()["ElectionType"],
    )
    df["number_eligible_voters"] = _round_to_uint32(raw["EligibleVoters"])
    df["number_valid_votes"] = _round_to_uint32(raw["Valid"])
    df["number_parties_effective"] = raw["HHI"]
    df["number_votes_far_right"] = _round_to_uint32(raw["Far_Right"])
    df["number_votes_far_left"] = _round_to_uint32(raw["Far_Left"])
    df["share_votes_far_right"] = raw["Far_Right_share"]
    df["share_votes_far_left"] = raw["Far_Left_share"]
    df["share_votes_far_any"] = raw["Far_share"]
    df["share_voter_turnout"] = raw["Turnout"]
    df["number_votes_far_any_incumbent"] = _round_to_uint32(raw["F0Far_Incumbent"])
    df["pm_party_orientation"] = _pm_party_orientation(raw["left"])
    return df


def _convert_election_cats(raw_sr, labels_dict):
    election_cats = labels_dict.copy()
    for i, to_append in (5, " A"), (7, " B"):
        election_cats[i] += to_append
        sr = raw_sr.astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
    return sr.cat.rename_categories(election_cats)


def _round_to_uint32(sr):
    return sr.round().astype(pd.UInt32Dtype())


def _pm_party_orientation(pm_party_left):
    sr = pm_party_left.astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
    return sr.cat.rename_categories(
        {0: "Other", 1: "Left-leaning"},
    )


raw = pd.read_stata(data_file, convert_categoricals=False)
metadata = pd.io.stata.StataReader(data_file)

cleaned_data = clean_data(raw=raw, metadata=metadata)

---
### Task 2 (spend max 10 min on this, else consider as bonus!)

Verify that the code you wrote in task 1 produces the same DataFrame as the code from
the solution to exercise 4.

> **Note:** Try first using `(data == other_data).all().all()`. Make sure you can run it
> without error. Explain why this becomes `False` (you will need to remove one or both 
> of the `.all()` method calls to find out). Then use `data.equals(other_data)` and
> explain why this works.

In [ ]:

(data == cleaned_data).all().all()

In [ ]:

data.equals(cleaned_data)


The first solution fails for missing values in floating point and categorical columns.
They compare as `False` against each other, the behaviour of `data.equals()` is
different in this respect.

The reason is that traditionally, pandas has been using `np.nan` for missing values. By
definition, these compare as `False` against each other. This makes sense for
computations, i.e., should the logarithm of zero be the same as dividing some number by
zero? This behaviour is not desirable for missing values, however. They are not there,
period.


## Running pytask

---
### Task 3

Run pytask in the current directory and find out what it does!


It executes a function which
- reads in data from `Alesina.xlsx`, 
- picks country, year, and two columns of interest with different deficit definitions,
- writes it to `bld/deficits.pkl`.

Find out via:
- `pytask collect --nodes`
- `pytask`
- inspecting source code and resulting directories


## Merging

---
### Task 4

Read in cleaned data with raw and primary deficits by country and year!

In [ ]:

def_data = pd.read_pickle(this_dir / "bld" / "deficits.pkl")

---
### Task 5

Your final goal is to create scatterplots with the following properties:

- x-axis: one of the three `share_far_*`-variables
- y-axis: one of the two deficit measures. 
- Colors of the dots given by NUTS region
- Readable names in the legend
- Focus on years with elections

To accomplish this task, you will need the following columns from the elections data:

In [ ]:
election_cols = [
    "country",
    "year",
    "nuts_name",
    "election_type",
    "share_votes_far_right",
    "share_votes_far_left",
    "share_votes_far_any",
]

Explain why you need each of those columns! 


- `country` - to merge with the deficit data
- `year` - to merge with the deficit data
- `nuts_name` - for colouring the dots in the scatterplot
- `election_type` - to select only years with elections
- `share_votes_far_right` - one of the axes in a plot
- `share_votes_far_left` - one of the axes in a plot
- `share_votes_far_any` - one of the axes in a plot


---
### Task 6

Think about how you would go about merging the elections data with the deficits data in
order to accomplish the goal outlined above.

Focus on the following dimension:
1. Merge key(s) (or column(s) to merge on)
2. Type of merge (inner, outer, left, right)


- we will want to merge on country and year because this is the level of aggregation in
  the deficit data.
- we will want to do an inner join. Election data only has European countries in it;
  there is no point in keeping countries that are not in these data. Similarly, it is
  pointless to keep years which might not be present in one of the datasets.

---
### Task 7

Check whether the contents of the merge keys that you identified are compatible in the
two datasets.

> **Hint:** One way to achieve this is to convert each of the columns' contents to a set
> and check whether their intersection is what you expect it to be!

In [ ]:

set(def_data["country"]).intersection(set(cleaned_data["country"]))

In [ ]:

set(def_data["year"]).intersection(set(cleaned_data["year"]))


Looks good, we can proceed.

---
### Task 8

Perform the merge operation that you identified, keeping only the columns
`election_cols` from the election data.

Assign the resulting object to a variable `final_data`.

In [ ]:
final_data = cleaned_data[election_cols].merge(
    right=def_data,
    how="inner",
    on=["country", "year"],
)

---
### Task 9

Make a scatterplot for years with elections that shows:
- the primary deficit on the x-axis
- the share of votes for any extremist parties on the y-axis
- the dots coloured by NUTS region

In [ ]:
only_elections = final_data[final_data["election_type"].notna()]
final_data.plot.scatter(x="primary_deficit", y="share_votes_far_any", color="country")

## Writing (py)tasks

---
### Task 10

Create a file `task_clean_election_data.py`, which contains a task function that reads
in the raw election data and writes out the cleaned version to the `bld` directory, to
a file called `election_results.pkl`.

Verify that this task is being run if you start pytask from the command line and that
the result is the same as the cleaned data from Task 1.

> **Hint:** You may find it easiest to 
> 1. copy `task_clean_alesina_data.py` and modify it.
> 2. copy and paste the functional code you wrote above

In [ ]:
!! solution

pd.read_pickle("bld/election_results.pkl").equals(cleaned_data)

---
### Task 11

Create a file `task_merge_data.py`, which contains a task function that reads in the two
cleaned datasets, keeps the `election_cols` from above, and saves the resulting
DataFrame in a file `merged_data.pkl` in the `bld` directory.

Verify that this task is being run if you start pytask from the command line and that
the result is the same as the merged data from Task 8.

In [ ]:
!! solution

pd.read_pickle("bld/merged_data.pkl").equals(final_data)

---
### Task 12

Create a file `task_create_scatterplots.py`, which contains a task function that reads
in the merged data and creates 6 scatterplots similar to the one above. They should vary
along what is depicted along their x-axes and y-axes:

- the two deficit measures
- the three `share_far_*`-variables

In all cases, dots should be coloured by the NUTS region. Give the files suitable names.

> *Note:* On some Windows computers, the exporting of plotly figures fails. See this
> [issue here](https://github.com/plotly/Kaleido/issues/134). If you experience that
> problem, please add a post to that issue. It would be really helpful if they fixed
> this, the more people complain, the more likely they are to do so.
>
> In case this affects you, try the workaround described
> [here](https://effective-programming-practices.vercel.app/plotting/why_plotly_prerequisites/objectives_materials.html#windows-workaround)